In [1]:
!pip install pymongo langchain-huggingface langchain-mongodb python-dotenv tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00


In [2]:
import json
from google.colab import userdata
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_core.documents import Document
from dotenv import load_dotenv

In [13]:
client = MongoClient(
    userdata.get("MONGO_DB_URI").strip(),

)
collection = client["rag_db"]["chunks"]

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [6]:
with open("/content/complete_windows.json") as f:
    data = json.load(f)

documents = [
    Document(
        page_content=item.get("Content", ""),
        metadata={"source": item.get("source", ""), "id": item.get("id", "")}
    )
    for item in data
]

In [14]:
from tqdm import tqdm
BATCH_SIZE=5
vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embeddings,
    index_name="vector_index"
)
for i in tqdm((0,len(documents),BATCH_SIZE),desc=f"Ingesting batches"):
  vector_store.add_documents(documents[i:i+BATCH_SIZE])
print(f"Inserted {len(documents)} documents.")

Ingesting batches: 100%|██████████| 3/3 [00:02<00:00,  1.30it/s]

Inserted 469 documents.


In [17]:
existing = list(collection.list_search_indexes())

if any(idx["name"] == "vector_index" for idx in existing):
    print("Index already exists!")
else:
    index_model = SearchIndexModel(
        definition={
            "fields": [
                {
                    "type": "vector",
                    "path": "embedding",
                    "numDimensions": 1024,
                    "similarity": "cosine"
                }
            ]
        },
        name="vector_index",
        type="vectorSearch"
    )
    collection.create_search_index(model=index_model)
    print("Index creation started — takes ~30-60s to become queryable.")

Index creation started — takes ~30-60s to become queryable.
